In [1]:
import pandas as pd
from src.data.ingestion import load_tickets

In [2]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from scipy.sparse import csr_matrix, hstack

from sentence_transformers import SentenceTransformer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder


from xgboost import XGBClassifier

/home/ramesh/Personal_projects/AI-based-Product-Support-System/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
json_data = load_tickets("../data/raw/support_tickets.json")

In [4]:
from src.classification.feature_builder import BuildClassificationFeatures


In [6]:
def train_mode_for_category_prediction(json_data):
    features = BuildClassificationFeatures(
    json_data[0],
    label_mode="category",
    text_mode='tfidf'
        ).build(sample_sizes = None)
    
        # --- XGBoost ---
    xgb = XGBClassifier(
        objective="multi:softprob",
        #num_class=len(le.classes_),
        n_estimators=500,
        max_depth=6,
        learning_rate=0.08,
        subsample=0.9,
        colsample_bytree=0.9,
        reg_lambda=1.0,
        tree_method="hist",
        eval_metric="mlogloss",
        random_state=42,
    )

    xgb.fit(features.X_train, features.y_train)

        # VAL
    val_pred_enc = xgb.predict(features.X_val)
    val_pred = features.label_encoder.inverse_transform(val_pred_enc)
    val_original = features.label_encoder.inverse_transform(features.y_val)

    print("VAL weighted F1:", round(f1_score(val_original, val_pred, average="weighted"), 4))
    print(classification_report(val_original, val_pred, digits=3))

    # TEST
    test_pred_enc = xgb.predict(features.X_test)
    test_pred = features.label_encoder.inverse_transform(test_pred_enc)
    tesr_original = features.label_encoder.inverse_transform(features.y_test)

    print("TEST weighted F1:", round(f1_score(tesr_original, test_pred, average="weighted"), 4))
    print(classification_report(tesr_original, test_pred, digits=3))
    return features, xgb


In [7]:
features, xgb = train_mode_for_category_prediction(json_data)

VAL weighted F1: 1.0
                 precision    recall  f1-score   support

     Data Issue      1.000     1.000     1.000      2811
Feature Request      1.000     1.000     1.000      3348
       Security      1.000     1.000     1.000      2538
Technical Issue      1.000     1.000     1.000      2325

       accuracy                          1.000     11022
      macro avg      1.000     1.000     1.000     11022
   weighted avg      1.000     1.000     1.000     11022

TEST weighted F1: 1.0
                 precision    recall  f1-score   support

     Data Issue      1.000     1.000     1.000      1872
Feature Request      1.000     1.000     1.000      3349
       Security      1.000     1.000     1.000      2979
Technical Issue      1.000     1.000     1.000      3305

       accuracy                          1.000     11505
      macro avg      1.000     1.000     1.000     11505
   weighted avg      1.000     1.000     1.000     11505



In [10]:
features.label_encoder.classes_

array(['Account Management', 'Data Issue', 'Feature Request', 'Security',
       'Technical Issue'], dtype=object)

In [14]:
#features.embedder.get_feature_names_out()

In [16]:
features.ohe.categories_

[array(['API Gateway', 'Analytics Dashboard', 'CloudBackup Enterprise',
        'DataSync Pro', 'StreamProcessor'], dtype=object),
 array(['api_connector', 'auth_service', 'backup_service',
        'batch_processor', 'cache_layer', 'compression_engine',
        'data_aggregator', 'data_validator', 'encryption_layer',
        'error_handler', 'event_handler', 'export_module', 'monitoring',
        'rate_limiter', 'report_builder', 'request_router',
        'restore_module', 'scheduler', 'sync_engine', 'visualization'],
       dtype=object),
 array(['critical', 'high', 'low', 'medium'], dtype=object),
 array(['api', 'chat', 'email', 'phone', 'portal', 'slack'], dtype=object),
 array(['enterprise', 'free', 'premium', 'professional', 'starter'],
       dtype=object)]

### Model for each category

In [5]:
from xgboost import XGBClassifier
from sklearn.metrics import classification_report, f1_score

from src.classification.save_load import save_category_artifacts

def create_model_for_each_category(json_data):
    categories = ['Feature Request', 'Account Management', 'Security']

    for category in categories:
        filtered_tickets = [t for t in json_data[0] if t.category == category]
        print(f"number of tickets in category {category}: {len(filtered_tickets)}")

        builder = BuildClassificationFeatures(
            filtered_tickets,
            label_mode="subcategory",
            text_mode="tfidf",     # or "transformer"
        )

        features = builder.build(sample_sizes=None)

        # Re-init model per category (recommended)
        xgb = XGBClassifier(
            n_estimators=500,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.9,
            colsample_bytree=0.9,
            tree_method="hist",
            n_jobs=-1,
            random_state=42,
        )

        xgb.fit(features.X_train, features.y_train)

        # VAL
        val_pred_enc = xgb.predict(features.X_val)
        val_pred = features.label_encoder.inverse_transform(val_pred_enc)
        val_true = features.label_encoder.inverse_transform(features.y_val)

        print("VAL weighted F1:", round(f1_score(val_true, val_pred, average="weighted"), 4))
        print(classification_report(val_true, val_pred, digits=3))

        # TEST
        test_pred_enc = xgb.predict(features.X_test)
        test_pred = features.label_encoder.inverse_transform(test_pred_enc)
        test_true = features.label_encoder.inverse_transform(features.y_test)

        print("TEST weighted F1:", round(f1_score(test_true, test_pred, average="weighted"), 4))
        print(classification_report(test_true, test_pred, digits=3))

        # SAVE artifacts per category
        save_category_artifacts(
            category=category,
            model=xgb,
            features=features,
            text_mode=builder.text_mode,
            out_dir="trained_models",
            transformer_model_name=getattr(builder, "model_name", None),
            transformer_batch_size=getattr(builder, "batch_size", None),
            transformer_device=getattr(builder, "device", None),
            normalize_embeddings=getattr(builder, "normalize_embeddings", None),
        )

create_model_for_each_category(json_data)


number of tickets in category Feature Request: 22047
VAL weighted F1: 0.198
               precision    recall  f1-score   support

          API      0.220     0.210     0.215       691
Documentation      0.192     0.252     0.218       650
  Enhancement      0.183     0.168     0.175       660
  New Feature      0.196     0.095     0.128       687
        UI/UX      0.220     0.295     0.252       708

     accuracy                          0.204      3396
    macro avg      0.203     0.204     0.198      3396
 weighted avg      0.203     0.204     0.198      3396

TEST weighted F1: 0.1914
               precision    recall  f1-score   support

          API      0.200     0.179     0.189       708
Documentation      0.196     0.248     0.219       661
  Enhancement      0.201     0.179     0.190       669
  New Feature      0.191     0.092     0.124       661
        UI/UX      0.198     0.292     0.236       650

     accuracy                          0.198      3349
    macro avg 